
# 26A R2 — Frozen V5 Candidate vs Production Control

**R2 fix only:** the previous notebook re-merged `positive_earlier_calc`,
`preassigned_axis`, and `collection_wave`, causing pandas `_x/_y` suffixes.
R2 merges only metadata not already present and hard-checks lineage.

No change to:
- frozen candidate
- 193 PRIMARY pairs
- metric
- bootstrap
- Control definition
- gates
- CONFIRM policy


In [1]:

from pathlib import Path
from datetime import datetime
import hashlib, json, sys, warnings
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")
NOTEBOOK_VERSION="SAJU_ML_V5_CANDIDATE_VS_CONTROL_26A_R2_MERGE_FIX_20260817"
SEED=20260817
N_BOOTSTRAP=10000

def repo_root(start=None):
    p=Path(start or Path.cwd()).resolve()
    for c in [p]+list(p.parents):
        if (c/"saju_engine.py").exists():
            return c
    raise FileNotFoundError("Run inside Chartpalja repository.")

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,"rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""):
            h.update(chunk)
    return h.hexdigest()

ROOT=repo_root()
A25=ROOT/"research/ml/artifacts/v5_tg10_multitask_balanced"
A24=ROOT/"research/ml/artifacts/v5_astrology_discovery_tournament"

DEC25=A25/"V5_25A_TG10_BALANCED_MULTITASK_DECISION.json"
SPEC25=A25/"V5_25A_FROZEN_CANDIDATE_MODEL_SPEC.json"
COEF25=A25/"V5_25A_FROZEN_CANDIDATE_COEFFICIENTS.csv"
OOF25=A25/"V5_25A_PRIMARY_OOF_PAIR_SCORES.csv"

PAIR=A24/"V5_DISCOVERY_PAIR_DIFF_PRIMARY.csv"
ENGINE_INPUT=A24/"V5_DISCOVERY_ENGINE_INPUT_AUDIT.csv"
PREREG=ROOT/"research/ml_corpus/v5_ground_truth/V5_FROZEN_CANDIDATE_VS_PRODUCTION_CONTROL_PREREGISTRATION.json"

OUT=ROOT/"research/ml/artifacts/v5_candidate_vs_control"
OUT.mkdir(parents=True,exist_ok=True)

for p in [DEC25,SPEC25,COEF25,OOF25,PAIR,ENGINE_INPUT,PREREG]:
    if not p.exists(): raise FileNotFoundError(p)

dec=json.load(open(DEC25,encoding="utf-8"))
spec=json.load(open(SPEC25,encoding="utf-8"))
protocol=json.load(open(PREREG,encoding="utf-8"))

assert dec["status"]=="V5_TG10_BALANCED_GENERAL_CANDIDATE_FROZEN_READY_FOR_CONTROL_BENCHMARK"
assert dec["winner"]=="TG10_MULTITASK_EFFECT_RIDGE_BALANCED"
assert spec["status"]=="FROZEN_BEFORE_CONTROL_AND_CONFIRM"
assert spec["architecture"]==dec["winner"]
assert spec["coefficients_sha256"]==sha256_file(COEF25)
assert spec["axis_required_at_inference"] is False
assert spec["control_scored"] is False
assert spec["confirm_loaded"] is False
assert protocol["status"]=="PREDECLARED_AFTER_25A_CANDIDATE_FREEZE_BEFORE_CONTROL_SCORING"

pairs=pd.read_csv(PAIR)
oof=pd.read_csv(OOF25)
eng=pd.read_csv(ENGINE_INPUT)

assert pairs.pair_id.nunique()==len(pairs)==193
winner=dec["winner"]
cand=oof[(oof.model==winner)&(oof.score_kind=="GENERAL")].copy()
assert cand.pair_id.nunique()==193
assert cand["repeat"].nunique()==5

CONFIRM_LOADED=False
print("Frozen candidate:",winner)
print("PRIMARY:",len(pairs),"pairs /",pairs.subject_id.nunique(),"subjects")
print("Candidate OOF repeats:",cand["repeat"].nunique())
print("CONFIRM loaded:",CONFIRM_LOADED)


Frozen candidate: TG10_MULTITASK_EFFECT_RIDGE_BALANCED
PRIMARY: 193 pairs / 100 subjects
Candidate OOF repeats: 5
CONFIRM loaded: False


## 1. Reproduce candidate OOF metric before opening Control

In [2]:

def chronology_balanced_metric(frame,correct_col):
    vals=[]
    for ori in [0,1]:
        h=frame[frame["positive_earlier_calc"]==ori]
        if h.empty: return np.nan
        vals.append(h.groupby("subject_id")[correct_col].mean().mean())
    return float(np.mean(vals))

def orientation_metrics(frame,correct_col):
    out={}
    for ori in [0,1]:
        h=frame[frame["positive_earlier_calc"]==ori]
        out[ori]=float(h.groupby("subject_id")[correct_col].mean().mean()) if len(h) else np.nan
    return out

# R2: cand already has chronology / axis / wave. Merge only missing metadata.
meta=pairs[["pair_id","subject_id","name","positive_year","negative_year"]].copy()

cand2=cand.merge(
    meta,on=["pair_id","subject_id"],
    how="left",validate="many_to_one"
)

# Hard lineage assertions against frozen pair file.
pair_check=pairs[[
    "pair_id","subject_id","positive_earlier_calc",
    "preassigned_axis","collection_wave"
]].copy()

chk=cand2[[
    "pair_id","subject_id","positive_earlier_calc",
    "preassigned_axis","collection_wave"
]].drop_duplicates().merge(
    pair_check,
    on=["pair_id","subject_id"],
    how="left",
    suffixes=("_oof","_pair"),
    validate="one_to_one"
)

assert (chk["positive_earlier_calc_oof"].astype(int)==chk["positive_earlier_calc_pair"].astype(int)).all()
assert (chk["preassigned_axis_oof"].astype(str)==chk["preassigned_axis_pair"].astype(str)).all()
assert (chk["collection_wave_oof"].astype(str)==chk["collection_wave_pair"].astype(str)).all()

rep=[]
for repeat,g in cand2.groupby("repeat"):
    ori=orientation_metrics(g,"correct")
    rep.append({
        "repeat":int(repeat),
        "balanced_macro":chronology_balanced_metric(g,"correct"),
        "positive_later":ori[0],
        "positive_earlier":ori[1]
    })
rep=pd.DataFrame(rep)
candidate_reproduced=float(rep.balanced_macro.mean())

frozen_gate=[x for x in dec["gates"] if x["model"]==winner][0]
assert abs(candidate_reproduced-float(frozen_gate["primary_balanced"]))<1e-12

print("Candidate OOF metric reproduced:",candidate_reproduced)
display(rep)


Candidate OOF metric reproduced: 0.6767721518987342


,repeat,balanced_macro,positive_later,positive_earlier
0,0,0.707637,0.760000,0.655274
1,1,0.669346,0.710000,0.628692
2,2,0.655169,0.700000,0.610338
3,3,0.689641,0.736667,0.642616
4,4,0.662068,0.710000,0.614135


## 2. Score current Production Control once

In [3]:

sys.path.insert(0,str(ROOT))
import saju_engine as se
import sajupy

def parse_utc_offset(raw):
    if isinstance(raw,(int,float)): return float(raw)
    s=str(raw).strip()
    sign=-1.0 if s.startswith("-") else 1.0
    s=s[1:] if s[:1] in "+-" else s
    hh,mm=s.split(":")
    return sign*(int(hh)+int(mm)/60.0)

def public_lon_corrected_birth(public_birth):
    y,m,d=map(int,public_birth["date"].split("-"))
    hh,mi=map(int,public_birth["time"].split(":")[:2])
    lon=float(public_birth["longitude"])
    utc_hours=parse_utc_offset(public_birth["utc_offset"])
    calc=sajupy.get_saju_calculator()
    civil=datetime(y,m,d,hh,mi)
    corr=float(calc._calculate_solar_time_correction(lon,utc_hours))
    h2,min2,dc=calc._adjust_time_for_solar(civil.hour,civil.minute,corr)
    y2,m2,d2=calc._adjust_date_for_solar(civil.year,civil.month,civil.day,dc)
    return {"y":y2,"m":m2,"d":d2,"h":h2,"min":min2}

def compute_timeline(subject):
    corrected=public_lon_corrected_birth({
        "date":str(subject["birth_date"]),
        "time":str(subject["birth_time"]),
        "utc_offset":str(subject["utc_offset"]),
        "longitude":float(subject["longitude"])
    })
    inp=se.BirthInput(
        year=int(corrected["y"]),month=int(corrected["m"]),day=int(corrected["d"]),
        hour=int(corrected["h"]),minute=int(corrected["min"]),
        gender=str(subject["gender"]),calendar="solar",is_leap_month=False,
        use_solar_time=False,utc_offset=9
    )
    result=se.compute_all(inp)
    return {int(r["year"]):r for r in result["chart_data"]["연도별_타임라인"]}

def control_score(meta_row):
    candle=meta_row.get("candle") or {}
    v=candle.get("close")
    return float(v) if v is not None else np.nan

subject_map={str(r.subject_id):r.to_dict() for _,r in eng.iterrows()}
needed=set(pairs.subject_id.astype(str))
assert needed.issubset(subject_map)

timeline_cache={}; fail=[]
for i,sid in enumerate(sorted(needed)):
    try:
        timeline_cache[sid]=compute_timeline(subject_map[sid])
    except Exception as e:
        fail.append({"subject_id":sid,"error":repr(e)})
    if (i+1)%10==0: print("Control subjects:",i+1,"/",len(needed))

pd.DataFrame(fail).to_csv(OUT/"V5_CONTROL_ENGINE_FAILURES.csv",index=False)
if fail:
    raise RuntimeError(f"Control scoring failed for {len(fail)} subjects.")

rows=[]
for _,r in pairs.iterrows():
    sid=str(r.subject_id); tl=timeline_cache[sid]
    py=int(r.positive_year); ny=int(r.negative_year)
    if py not in tl or ny not in tl:
        raise RuntimeError(f"Control timeline missing: {sid} {py}/{ny}")
    ps=control_score(tl[py]); ns=control_score(tl[ny])
    if not np.isfinite(ps) or not np.isfinite(ns):
        raise RuntimeError(f"Non-finite Control score: {sid} {py}/{ny}")
    corr=1.0 if ps>ns else (0.0 if ps<ns else 0.5)
    rows.append({
        "pair_id":r.pair_id,"subject_id":sid,
        "preassigned_axis":r.preassigned_axis,
        "collection_wave":r.collection_wave,
        "positive_earlier_calc":int(r.positive_earlier_calc),
        "positive_year":py,"negative_year":ny,
        "control_positive_score":ps,
        "control_negative_score":ns,
        "control_correct":corr
    })

control=pd.DataFrame(rows)
control.to_csv(OUT/"V5_PRODUCTION_CONTROL_PAIR_SCORES.csv",index=False)
print("Control scoring complete:",len(control))


[SAJU_DEBUG] original_input: 1963-07-04 07:54
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1963-07-04
[SAJU_DEBUG] final_datetime(KST): 1963-07-04T07:54:00+09:00
[SAJU_DEBUG] pillars: 연=癸卯 월=戊午 일=戊申 시=丙辰
[SAJU_DEBUG] original_input: 1913-01-09 21:43
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1913-01-09
[SAJU_DEBUG] final_datetime(KST): 1913-01-09T21:43:00+09:00
[SAJU_DEBUG] pillars: 연=壬子 월=癸丑 일=庚寅 시=丁亥
[SAJU_DEBUG] original_input: 1921-02-24 01:17
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1921-02-24
[SAJU_DEBUG] final_datetime(KST): 1921-02-24T01:17:00+09:00
[SAJU_DEBUG] 반시보정: 癸丑→壬子 (01:17)
[SAJU_DEBUG] pillars: 연=辛酉 월=庚寅 일=戊午 시=壬子
[SAJU_DEBUG] original_input: 1924-10-01 06:22
[SAJU_DEBUG] calendar=solar, is_l

Control subjects: 10 / 100


[SAJU_DEBUG] original_input: 1902-04-18 04:32
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1902-04-18
[SAJU_DEBUG] final_datetime(KST): 1902-04-18T04:32:00+09:00
[SAJU_DEBUG] pillars: 연=壬寅 월=甲辰 일=辛未 시=庚寅
[SAJU_DEBUG] original_input: 1900-01-08 13:00
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1900-01-08
[SAJU_DEBUG] final_datetime(KST): 1900-01-08T13:00:00+09:00
[SAJU_DEBUG] 반시보정: 乙未→甲午 (13:00)
[SAJU_DEBUG] pillars: 연=己亥 월=丁丑 일=辛巳 시=甲午
[SAJU_DEBUG] original_input: 1979-02-16 18:10
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1979-02-16
[SAJU_DEBUG] final_datetime(KST): 1979-02-16T18:10:00+09:00
[SAJU_DEBUG] pillars: 연=己未 월=丙寅 일=甲寅 시=癸酉
[SAJU_DEBUG] original_input: 1986-09-01 02:59
[SAJU_DEBUG] calendar=solar, is_l

Control subjects: 20 / 100


[SAJU_DEBUG] original_input: 1906-10-08 23:57
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1906-10-08
[SAJU_DEBUG] final_datetime(KST): 1906-10-08T23:57:00+09:00
[SAJU_DEBUG] pillars: 연=丙午 월=戊戌 일=丙戌 시=戊子
[SAJU_DEBUG] original_input: 1929-04-19 16:35
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1929-04-19
[SAJU_DEBUG] final_datetime(KST): 1929-04-19T16:35:00+09:00
[SAJU_DEBUG] pillars: 연=己巳 월=戊辰 일=甲午 시=壬申
[SAJU_DEBUG] original_input: 1913-02-18 21:44
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1913-02-18
[SAJU_DEBUG] final_datetime(KST): 1913-02-18T21:44:00+09:00
[SAJU_DEBUG] pillars: 연=癸丑 월=甲寅 일=庚午 시=丁亥
[SAJU_DEBUG] original_input: 1941-12-31 15:12
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJ

...

Control subjects: 80 / 100


[SAJU_DEBUG] original_input: 1944-10-29 06:40
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1944-10-29
[SAJU_DEBUG] final_datetime(KST): 1944-10-29T06:40:00+09:00
[SAJU_DEBUG] pillars: 연=甲申 월=甲戌 일=丙寅 시=辛卯
[SAJU_DEBUG] original_input: 1977-04-23 09:13
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1977-04-23
[SAJU_DEBUG] final_datetime(KST): 1977-04-23T09:13:00+09:00
[SAJU_DEBUG] 반시보정: 辛巳→庚辰 (09:13)
[SAJU_DEBUG] pillars: 연=丁巳 월=甲辰 일=庚戌 시=庚辰
[SAJU_DEBUG] original_input: 1947-09-20 12:03
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=female
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1947-09-20
[SAJU_DEBUG] final_datetime(KST): 1947-09-20T12:03:00+09:00
[SAJU_DEBUG] pillars: 연=丁亥 월=己酉 일=壬寅 시=丙午
[SAJU_DEBUG] original_input: 1918-03-20 12:27
[SAJU_DEBUG] calendar=solar, is

Control subjects: 90 / 100


[SAJU_DEBUG] original_input: 1944-04-07 20:36
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1944-04-07
[SAJU_DEBUG] final_datetime(KST): 1944-04-07T20:36:00+09:00
[SAJU_DEBUG] pillars: 연=甲申 월=戊辰 일=辛丑 시=戊戌
[SAJU_DEBUG] original_input: 1973-07-30 21:40
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1973-07-30
[SAJU_DEBUG] final_datetime(KST): 1973-07-30T21:40:00+09:00
[SAJU_DEBUG] pillars: 연=癸丑 월=己未 일=丁卯 시=辛亥
[SAJU_DEBUG] original_input: 1906-03-19 18:58
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJU_DEBUG] early_zi_time=False
[SAJU_DEBUG] solar input (no conversion): 1906-03-19
[SAJU_DEBUG] final_datetime(KST): 1906-03-19T18:58:00+09:00
[SAJU_DEBUG] pillars: 연=丙午 월=辛卯 일=壬戌 시=己酉
[SAJU_DEBUG] original_input: 1945-10-13 05:37
[SAJU_DEBUG] calendar=solar, is_leap_month=False, gender=male
[SAJ

Control subjects: 100 / 100
Control scoring complete: 193


## 3. Headline candidate OOF vs Control

In [4]:

cavg=(
    cand2.groupby([
        "pair_id","subject_id","positive_earlier_calc",
        "preassigned_axis","collection_wave"
    ]).correct.mean().reset_index(name="candidate_oof_correct")
)

cmp=cavg.merge(
    control[["pair_id","subject_id","control_correct"]],
    on=["pair_id","subject_id"],how="inner",validate="one_to_one"
)
assert len(cmp)==193

cand_metric=chronology_balanced_metric(cmp,"candidate_oof_correct")
control_metric=chronology_balanced_metric(cmp,"control_correct")
cand_ori=orientation_metrics(cmp,"candidate_oof_correct")
ctrl_ori=orientation_metrics(cmp,"control_correct")

headline=pd.DataFrame([{
    "candidate":winner,
    "candidate_balanced_macro":cand_metric,
    "control_balanced_macro":control_metric,
    "delta_candidate_minus_control":cand_metric-control_metric,
    "candidate_positive_later":cand_ori[0],
    "control_positive_later":ctrl_ori[0],
    "candidate_positive_earlier":cand_ori[1],
    "control_positive_earlier":ctrl_ori[1],
}])
headline.to_csv(OUT/"V5_CANDIDATE_VS_CONTROL_HEADLINE.csv",index=False)
display(headline)


,candidate,candidate_balanced_macro,control_balanced_macro,delta_candidate_minus_control,candidate_positive_later,control_positive_later,candidate_positive_earlier,control_positive_earlier
0,TG10_MULTITASK_EFFECT_RIDGE_BALANCED,0.676772,0.425306,0.251466,0.723333,0.385,0.630211,0.465612


## 4. Paired subject bootstrap

In [5]:

subjects=sorted(cmp.subject_id.unique())
ix={s:i for i,s in enumerate(subjects)}

def subject_orientation_matrix(col):
    M=np.full((len(subjects),2),np.nan)
    z=cmp.groupby(["subject_id","positive_earlier_calc"])[col].mean().reset_index()
    for _,r in z.iterrows():
        M[ix[r.subject_id],int(r.positive_earlier_calc)]=float(r[col])
    return M

C=subject_orientation_matrix("candidate_oof_correct")
K=subject_orientation_matrix("control_correct")

def metric_with_counts(M,counts):
    vals=[]
    for ori in [0,1]:
        mask=~np.isnan(M[:,ori]); den=counts[mask].sum()
        vals.append(np.sum(M[mask,ori]*counts[mask])/den if den else np.nan)
    return float(np.nanmean(vals))

rng=np.random.default_rng(SEED)
ds=np.empty(N_BOOTSTRAP)
for b in range(N_BOOTSTRAP):
    sample=rng.integers(0,len(subjects),len(subjects))
    counts=np.bincount(sample,minlength=len(subjects))
    ds[b]=metric_with_counts(C,counts)-metric_with_counts(K,counts)

boot=pd.DataFrame([{
    "comparison":"FROZEN_V5_OOF_minus_PRODUCTION_CONTROL",
    "observed_delta":cand_metric-control_metric,
    "bootstrap_mean_delta":float(ds.mean()),
    "ci025":float(np.quantile(ds,.025)),
    "ci975":float(np.quantile(ds,.975)),
    "p_delta_gt_0":float((ds>0).mean())
}])
boot.to_csv(OUT/"V5_CANDIDATE_VS_CONTROL_BOOTSTRAP.csv",index=False)
display(boot)


,comparison,observed_delta,bootstrap_mean_delta,ci025,ci975,p_delta_gt_0
0,FROZEN_V5_OOF_minus_PRODUCTION_CONTROL,0.251466,0.250983,0.146796,0.35147,1.0


## 5. Wave and axis diagnostics

In [6]:

def grouped_compare(frame,dim):
    rows=[]
    for key,g in frame.groupby(dim):
        co=orientation_metrics(g,"candidate_oof_correct")
        ko=orientation_metrics(g,"control_correct")
        cm=chronology_balanced_metric(g,"candidate_oof_correct")
        km=chronology_balanced_metric(g,"control_correct")
        rows.append({
            dim:key,
            "n_pairs":len(g),"n_subjects":g.subject_id.nunique(),
            "candidate_balanced":cm,"control_balanced":km,
            "delta_candidate_minus_control":cm-km,
            "candidate_positive_later":co[0],"control_positive_later":ko[0],
            "candidate_positive_earlier":co[1],"control_positive_earlier":ko[1],
        })
    return pd.DataFrame(rows)

wave=grouped_compare(cmp,"collection_wave")
axis=grouped_compare(cmp,"preassigned_axis")
wave.to_csv(OUT/"V5_CANDIDATE_VS_CONTROL_WAVE.csv",index=False)
axis.to_csv(OUT/"V5_CANDIDATE_VS_CONTROL_AXIS.csv",index=False)
display(wave); display(axis)


,collection_wave,n_pairs,n_subjects,candidate_balanced,control_balanced,delta_candidate_minus_control,candidate_positive_later,control_positive_later,candidate_positive_earlier,control_positive_earlier
0,E1,57,29,0.721932,0.374275,0.347657,0.784444,0.266667,0.659420,0.481884
1,E2,23,18,0.617647,0.491176,0.126471,0.600000,0.600000,0.635294,0.382353
2,ORIGINAL,113,53,0.662051,0.450321,0.211731,0.713333,0.408333,0.610769,0.492308


,preassigned_axis,n_pairs,n_subjects,candidate_balanced,control_balanced,delta_candidate_minus_control,candidate_positive_later,control_positive_later,candidate_positive_earlier,control_positive_earlier
0,COMPETITIVE,134,57,0.687654,0.443496,0.244158,0.726852,0.416667,0.648455,0.470325
1,PROJECT,12,9,0.540000,0.491667,0.048333,0.780000,0.400000,0.300000,0.583333
2,STATUS,47,34,0.673264,0.343750,0.329514,0.677778,0.250000,0.668750,0.437500


## 6. Frozen gate decision

In [7]:

g=protocol["promotion_gates"]
delta=float(headline.iloc[0].delta_candidate_minus_control)
pgt=float(boot.iloc[0].p_delta_gt_0)

later_deficit=float(headline.iloc[0].candidate_positive_later-headline.iloc[0].control_positive_later)
earlier_deficit=float(headline.iloc[0].candidate_positive_earlier-headline.iloc[0].control_positive_earlier)

supported=wave[wave.n_subjects>=int(g["supported_wave_min_subjects"])]
wave_min_delta=float(supported.delta_candidate_minus_control.min()) if len(supported) else np.nan

passes={
    "primary_delta_ge_001":delta>=float(g["primary_balanced_delta_vs_control_min"]),
    "bootstrap_p_ge_080":pgt>=float(g["bootstrap_probability_delta_gt_0_min"]),
    "positive_later_not_worse_gt_002":
        later_deficit>=-float(g["positive_later_candidate_not_worse_than_control_by_more_than"]),
    "positive_earlier_not_worse_gt_002":
        earlier_deficit>=-float(g["positive_earlier_candidate_not_worse_than_control_by_more_than"]),
    "supported_wave_not_worse_gt_003":
        True if np.isnan(wave_min_delta) else
        wave_min_delta>=-float(g["supported_wave_max_allowed_deficit_vs_control"]),
}

if all(passes.values()):
    status="V5_FROZEN_CANDIDATE_BEATS_CONTROL_READY_FOR_CONFIRM_PROTOCOL"
    confirm_allowed=True
else:
    status="V5_FROZEN_CANDIDATE_DOES_NOT_CLEAR_CONTROL_GATE_KEEP_CONFIRM_SEALED"
    confirm_allowed=False

decision={
    "version":"V5_CANDIDATE_VS_PRODUCTION_CONTROL_DECISION_V1",
    "notebook_version":NOTEBOOK_VERSION,
    "created_at":datetime.now().isoformat(timespec="seconds"),
    "status":status,
    "candidate":winner,
    "candidate_evaluation":"25A subject-disjoint repeated OOF GENERAL",
    "control_definition":"current runtime annual timeline candle.close",
    "primary_metric":"chronology_balanced_subject_macro_pairwise_accuracy",
    "headline":headline.iloc[0].to_dict(),
    "bootstrap":boot.iloc[0].to_dict(),
    "supported_wave_min_delta":wave_min_delta,
    "gates":passes,
    "all_gates":bool(all(passes.values())),
    "lineage":{
        "25A_decision_sha256":sha256_file(DEC25),
        "25A_candidate_spec_sha256":sha256_file(SPEC25),
        "25A_coefficients_sha256":sha256_file(COEF25),
        "25A_oof_sha256":sha256_file(OOF25),
        "primary_pairs_sha256":sha256_file(PAIR),
        "saju_engine_py_sha256":sha256_file(ROOT/"saju_engine.py"),
        "preregistration_sha256":sha256_file(PREREG)
    },
    "rules":{
        "candidate_retuned_after_freeze":False,
        "full_primary_candidate_used_for_DEV_headline":False,
        "pair_membership_changed":False,
        "confirm_loaded_or_researched":False,
        "control_scored":True
    },
    "confirm_protocol_may_begin":confirm_allowed,
    "next_rule":(
        "If READY: freeze a one-shot CONFIRM80 event-research/scoring protocol before opening CONFIRM. "
        "If FAIL: keep CONFIRM sealed and do not retune using Control results."
    )
}
json.dump(decision,open(OUT/"V5_CANDIDATE_VS_PRODUCTION_CONTROL_DECISION.json","w",encoding="utf-8"),
          ensure_ascii=False,indent=2)
print(json.dumps({
    "status":status,
    "candidate_balanced":cand_metric,
    "control_balanced":control_metric,
    "delta":delta,
    "p_delta_gt_0":pgt,
    "all_gates":all(passes.values()),
    "confirm_protocol_may_begin":confirm_allowed
},ensure_ascii=False,indent=2))


{
  "status": "V5_FROZEN_CANDIDATE_BEATS_CONTROL_READY_FOR_CONFIRM_PROTOCOL",
  "candidate_balanced": 0.6767721518987342,
  "control_balanced": 0.42530590717299577,
  "delta": 0.2514662447257384,
  "p_delta_gt_0": 1.0,
  "all_gates": true,
  "confirm_protocol_may_begin": true
}



## Send back after `Kernel Restart → Run All`

Send:

```text
V5_CANDIDATE_VS_PRODUCTION_CONTROL_DECISION.json
V5_CANDIDATE_VS_CONTROL_HEADLINE.csv
V5_CANDIDATE_VS_CONTROL_BOOTSTRAP.csv
V5_CANDIDATE_VS_CONTROL_WAVE.csv
V5_CANDIDATE_VS_CONTROL_AXIS.csv
```

If Control generation fails, send:
`V5_CONTROL_ENGINE_FAILURES.csv`

Do not open CONFIRM yet.
